# Chapter 9 — Exercise Solutions## Bandits & Offline RL[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 9.1: LinUCB Contextual Bandit

In [ ]:
import numpy as np, matplotlib.pyplot as pltclass LinUCBArm:    """One arm of a LinUCB contextual bandit. Linear reward model."""    def __init__(self, d, alpha=1.0):        self.A = np.eye(d)      # d×d positive definite matrix (A = X^T X + I)        self.b = np.zeros(d)    # d-vector (b = X^T r)        self.alpha = alpha      # exploration parameter    def score(self, context):        A_inv = np.linalg.inv(self.A)        theta = A_inv @ self.b              # estimated coefficients        mean  = theta @ context             # predicted reward        ucb   = self.alpha * np.sqrt(context @ A_inv @ context)  # exploration bonus        return mean + ucb    def update(self, context, reward):        self.A += np.outer(context, context)        self.b += context * rewardclass LinUCBBandit:    """LinUCB with d-dimensional context and K arms."""    def __init__(self, K, d, alpha=1.0):        self.arms = [LinUCBArm(d, alpha) for _ in range(K)]    def select(self, context):        return np.argmax([arm.score(context) for arm in self.arms])    def update(self, arm_idx, context, reward):        self.arms[arm_idx].update(context, reward)# Experiment: 10 arms, 5-dimensional contextnp.random.seed(42)K, d, T = 10, 5, 2000true_theta = np.random.randn(K, d)  # true linear reward parametersdef get_reward(arm, context):    return np.dot(true_theta[arm], context) + np.random.normal(0, 0.3)def best_arm(context):    return np.argmax([np.dot(true_theta[k], context) for k in range(K)])# Compare LinUCB vs UCB1 (context-blind)linucb  = LinUCBBandit(K, d, alpha=0.5)ucb1_counts = np.ones(K); ucb1_means = np.zeros(K); ucb1_t = 0regrets_linucb, regrets_ucb1 = [], []cum_r_linucb = cum_r_ucb1 = 0for t in range(T):    context = np.random.randn(d)    best = best_arm(context)    optimal_r = get_reward(best, context)    # LinUCB    arm_l = linucb.select(context)    r_l = get_reward(arm_l, context)    linucb.update(arm_l, context, r_l)    cum_r_linucb += optimal_r - r_l    regrets_linucb.append(cum_r_linucb)    # UCB1 (context-blind — ignores context)    ucb1_t += 1    ucb_vals = ucb1_means + np.sqrt(2*np.log(ucb1_t+1)/ucb1_counts)    arm_u = np.argmax(ucb_vals)    r_u = get_reward(arm_u, context)    ucb1_counts[arm_u] += 1    ucb1_means[arm_u] += (r_u - ucb1_means[arm_u]) / ucb1_counts[arm_u]    cum_r_ucb1 += optimal_r - r_u    regrets_ucb1.append(cum_r_ucb1)fig,ax = plt.subplots(figsize=(10,4))ax.plot(regrets_linucb, label='LinUCB (uses context)',   color='seagreen', lw=2)ax.plot(regrets_ucb1,   label='UCB1 (context-blind)',    color='tomato',   lw=2)ax.set_xlabel('Step'); ax.set_ylabel('Cumulative Regret')ax.set_title('LinUCB vs UCB1: Effect of Using Context'); ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print(f"LinUCB final regret: {regrets_linucb[-1]:.1f}")print(f"UCB1   final regret: {regrets_ucb1[-1]:.1f}")print("LinUCB wins because reward depends on context — ignoring context = wasted information")# YOUR TURN: Add a hybrid: UCB1 that uses context as features via simple averaging# Does it close the gap to LinUCB?

### Solution 9.2: Optimal ε for ε-Greedy

In [ ]:
import numpy as np, matplotlib.pyplot as pltdef run_bandit(n_arms=10, epsilon=0.1, n_steps=1000, n_runs=200):    """Run epsilon-greedy bandit for n_runs and return mean reward over last 100 steps."""    final_rewards = []    for _ in range(n_runs):        true_means = np.random.normal(0, 1, n_arms)        counts = np.zeros(n_arms); means = np.zeros(n_arms)        rewards_run = []        for t in range(n_steps):            if np.random.random() < epsilon:                arm = np.random.randint(n_arms)            else:                arm = np.argmax(means)            r = np.random.normal(true_means[arm], 1)            counts[arm] += 1            means[arm] += (r - means[arm]) / counts[arm]            rewards_run.append(r)        final_rewards.append(np.mean(rewards_run[-100:]))    return np.mean(final_rewards)epsilons = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]results_10  = [run_bandit(n_arms=10,  epsilon=e) for e in epsilons]results_50  = [run_bandit(n_arms=50,  epsilon=e) for e in epsilons]results_var = [run_bandit(n_arms=10,  epsilon=e, n_runs=200) for e in epsilons]fig, ax = plt.subplots(figsize=(10,5))ax.plot(epsilons, results_10, 'o-', label='10 arms, std=1', color='steelblue', lw=2)ax.plot(epsilons, results_50, 's-', label='50 arms, std=1', color='tomato',   lw=2)ax.set_xscale('log')ax.set_xlabel('ε (log scale)'); ax.set_ylabel('Mean Reward (last 100 steps)')ax.set_title('Optimal ε for ε-Greedy Bandit'); ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()opt_10 = epsilons[np.argmax(results_10)]opt_50 = epsilons[np.argmax(results_50)]print(f"Optimal ε for 10 arms: {opt_10}")print(f"Optimal ε for 50 arms: {opt_50}")print("Key insight: more arms → more exploration needed → higher optimal ε")print("Reward variance doesn't change optimal ε much (exploration need is about arms, not noise)")

### Solution 9.3: Offline Policy Evaluation with IPS

In [ ]:
import numpy as npnp.random.seed(42)N_ARMS = 5TRUE_MEANS = np.array([0.2, 0.4, 0.6, 0.3, 0.5])# Simulate logged data from behaviour policy (ε-greedy, ε=0.3)def behaviour_policy(context, epsilon=0.3):    """ε-greedy with arm 2 as default best known."""    if np.random.random() < epsilon:        arm = np.random.randint(N_ARMS)        prob = epsilon/N_ARMS + (1-epsilon)*(arm==2)    else:        arm = 2  # best arm known to behaviour policy        prob = epsilon/N_ARMS + (1-epsilon)    return arm, prob# Target policy: always recommend arm 3def target_policy_prob(arm):    """Returns probability of target policy choosing `arm`."""    return 1.0 if arm == 3 else 0.0# Collect n_log interactionsn_log = 5000log_data = []for _ in range(n_log):    context = np.random.randn(3)    arm, behaviour_prob = behaviour_policy(context)    reward = np.random.normal(TRUE_MEANS[arm], 0.5)    log_data.append({'arm': arm, 'reward': reward, 'behaviour_prob': behaviour_prob})# True value of target policy (ground truth for comparison)true_target_value = TRUE_MEANS[3]# Naive estimator (just average reward — WRONG, biased to behaviour policy)naive_value = np.mean([d['reward'] for d in log_data])# IPS estimator: reweight by π_target / π_behaviourips_numerator   = 0ips_denominator = 0for d in log_data:    arm  = d['arm']    pi_t = target_policy_prob(arm)   # 1 if arm==3, else 0    pi_b = d['behaviour_prob']    weight = pi_t / pi_b if pi_b > 0 else 0    ips_numerator += weight * d['reward']    ips_denominator += weightips_value = ips_numerator / max(ips_denominator, 1e-9)print(f"True target policy value (arm 3): {true_target_value:.4f}")print(f"Naive estimator:                  {naive_value:.4f}  (biased to behaviour policy)")print(f"IPS estimator:                    {ips_value:.4f}  (corrects for selection bias)")print(f"IPS error: {abs(ips_value-true_target_value):.4f}")print("\nIPS is unbiased when behaviour_prob > 0 for all actions taken by target policy")print("Limitation: high variance when weights are large (π_target >> π_behaviour)")# YOUR TURN: Implement DR (Doubly Robust) estimator# DR = DM_estimate + IPS correction — lower variance than pure IPS